In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/Priyanshuchaudhary2425/ScamGuard

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Priyanshuchaudhary2425/ScamGuard)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
import os
import torch
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

MODEL_NAME = "Priyanshuchaudhary2425/ScamGuard"
# https://huggingface.co/Priyanshuchaudhary2425/ScamGuard   this is hosted URL

# Check if model is already cached locally
model_cache_path = os.path.join(os.path.expanduser("~"), ".cache", "huggingface", "hub", MODEL_NAME.replace("/", "_"))

if os.path.exists(model_cache_path):
    print("✅ Loading model from local cache...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, local_files_only=True)
else:
    print("🔄 Downloading model from Hugging Face Hub...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.eval()
print("✅ ScamGuard model ready")

# Map label ids to readable labels (adjust if your model uses different)
label_map = {0: "positive", 1: "spam"}

def predict(text: str):
    inputs = tokenizer(text, return_tensors="pt")
    start = time.time()
    with torch.inference_mode():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        confidence, predicted_class = torch.max(probs, dim=1)
    end = time.time()
    inference_time = (end - start) * 1000
    label = label_map[predicted_class.item()]
    return label, confidence.item(), round(inference_time, 2)


